In [ ]:
# Robust Prophet + Optuna tuning (defensive, notebook-friendly)
!pip install optuna
!pip install prophet pystan --upgrade

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

from prophet import Prophet
from prophet.diagnostics import cross_validation
import optuna
from sklearn.metrics import mean_absolute_error, mean_squared_error
import math, sys

np.random.seed(42)

def create_dataset():
    start_date = datetime(2018, 1, 1)
    n_days = 1500
    dates = [start_date + timedelta(days=i) for i in range(n_days)]
    t = np.arange(n_days)
    trend = 0.002 * t**2 + 0.5 * t + 50
    yearly_season = 15 * np.sin(2 * np.pi * t / 365)
    weekly_season = 8 * np.sin(2 * np.pi * t / 7)
    monthly_season = 5 * np.sin(2 * np.pi * t / 30)
    noise = np.random.normal(0, 3, n_days)
    values = trend + yearly_season + weekly_season + monthly_season + noise
    return pd.DataFrame({'ds': dates, 'y': values})

data = create_dataset()

train_size = int(0.8 * len(data))
train_data = data.iloc[:train_size].reset_index(drop=True)
test_data = data.iloc[train_size:].reset_index(drop=True)

def make_prophet_model(params=None):
    params = params or {}
    model = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=True,
        daily_seasonality=False,
        changepoint_prior_scale=params.get('changepoint_prior_scale', 0.05),
        seasonality_prior_scale=params.get('seasonality_prior_scale', 10.0),
        holidays_prior_scale=params.get('holidays_prior_scale', 10.0),
        seasonality_mode=params.get('seasonality_mode', 'additive'),
    )
    model.add_seasonality(name='monthly', period=30.0, fourier_order=5)
    return model

# Baseline
baseline_model = make_prophet_model()
baseline_model.fit(train_data)

def run_cross_validation(model, df, initial_days='500 days', period='60 days', horizon='90 days'):
    try:
        df_cv = cross_validation(model, initial=initial_days, period=period, horizon=horizon, parallel=None)
    except Exception as e:
        print("Cross-validation failed:", e, file=sys.stderr)
        return None, None, None, None

    mae = mean_absolute_error(df_cv['y'], df_cv['yhat'])
    rmse = math.sqrt(mean_squared_error(df_cv['y'], df_cv['yhat']))

    # Safer MAPE calculation
    with np.errstate(divide='ignore', invalid='ignore'):
        ape = np.abs((df_cv['y'] - df_cv['yhat']) / df_cv['y'])
        ape = ape[np.isfinite(ape)]
        mape = np.mean(ape) * 100 if len(ape) > 0 else float('inf')

    return mae, rmse, mape, df_cv

baseline_mae, baseline_rmse, baseline_mape, baseline_cv = run_cross_validation(baseline_model, train_data)

def objective_function(trial):
    params = {
        'changepoint_prior_scale': trial.suggest_float('changepoint_prior_scale', 0.001, 0.5, log=True),
        'seasonality_prior_scale': trial.suggest_float('seasonality_prior_scale', 0.01, 20.0, log=True),
        'holidays_prior_scale': trial.suggest_float('holidays_prior_scale', 0.01, 20.0, log=True),
        'seasonality_mode': trial.suggest_categorical('seasonality_mode', ['additive', 'multiplicative'])
    }

    model = make_prophet_model(params)
    split_idx = 800
    subset_train = train_data.iloc[:split_idx].reset_index(drop=True)
    val_data = train_data.iloc[split_idx:].reset_index(drop=True)

    try:
        model.fit(subset_train)
        future = pd.DataFrame({'ds': val_data['ds']})
        forecast = model.predict(future)
        rmse = math.sqrt(mean_squared_error(val_data['y'].values, forecast['yhat'].values))
        return rmse
    except Exception as e:
        # Log and return a large numeric penalty so Optuna steers away
        print(f"Trial failed: {e}", file=sys.stderr)
        return 1e6

study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective_function, n_trials=20, show_progress_bar=True)

# Safe extraction of best params
if len(study.trials) == 0 or study.best_trial is None:
    print("Optuna found no valid trials.", file=sys.stderr)
    best_params = {}
    best_value = None
else:
    best_params = study.best_trial.params
    best_value = study.best_trial.value

final_model = make_prophet_model(best_params)
final_model.fit(train_data)

optimized_mae, optimized_rmse, optimized_mape, optimized_cv = run_cross_validation(final_model, train_data)

future_test = pd.DataFrame({'ds': test_data['ds']})
forecast_test = final_model.predict(future_test)
test_predictions = forecast_test[['ds', 'yhat']].reset_index(drop=True)

test_mae = mean_absolute_error(test_data['y'], test_predictions['yhat'])
test_rmse = math.sqrt(mean_squared_error(test_data['y'], test_predictions['yhat']))

with np.errstate(divide='ignore', invalid='ignore'):
    ape_test = np.abs((test_data['y'] - test_predictions['yhat']) / test_data['y'])
    ape_test = ape_test[np.isfinite(ape_test)]
    test_mape = np.mean(ape_test) * 100 if len(ape_test) > 0 else float('inf')

# Safely compute improvements (guard against None)
def safe_improvement(base, opt):
    if base is None or opt is None or base == 0:
        return None
    return ((base - opt) / base) * 100

mae_improvement = safe_improvement(baseline_mae, optimized_mae)
rmse_improvement = safe_improvement(baseline_rmse, optimized_rmse)

# Printing with graceful fallbacks
def fmt(x):
    if x is None or (isinstance(x, float) and (np.isnan(x) or np.isinf(x))):
        return "N/A"
    return f"{x:.4f}"

print("DATASET CHARACTERISTICS")
print("Total observations:", len(data))
print("Training set size:", len(train_data))
print("Test set size:", len(test_data))

print("\nBAYESIAN OPTIMIZATION RESULTS")
print("Total trials completed:", len(study.trials))
if best_value is not None:
    print("Best trial number:", study.best_trial.number)
    print("Best validation RMSE:", round(best_value, 6))
else:
    print("Best trial: N/A")

print("\nPERFORMANCE COMPARISON")
print("+------------------+-----------+-----------+-----------+")
print("| Model            | MAE       | RMSE      | MAPE (%)  |")
print("+------------------+-----------+-----------+-----------+")
print(f"| Baseline (CV)    | {fmt(baseline_mae)}  | {fmt(baseline_rmse)}  | {fmt(baseline_mape)}   |")
print(f"| Optimized (CV)   | {fmt(optimized_mae)}  | {fmt(optimized_rmse)}  | {fmt(optimized_mape)}   |")
print(f"| Optimized (Test) | {fmt(test_mae)}  | {fmt(test_rmse)}  | {fmt(test_mape)}   |")
print("+------------------+-----------+-----------+-----------+")

if mae_improvement is None:
    print("\nMAE improvement: N/A")
else:
    print(f"\nMAE improvement: {mae_improvement:.2f}%")
if rmse_improvement is None:
    print("RMSE improvement: N/A")
else:
    print(f"RMSE improvement: {rmse_improvement:.2f}%")

# Plotting - only when cv outputs exist
plt.figure(figsize=(12, 8))

plt.subplot(2, 2, 1)
plt.plot(data['ds'], data['y'])
plt.title('Complete Dataset')
plt.xticks(rotation=45)

if baseline_cv is not None:
    plt.subplot(2, 2, 2)
    plt.plot(baseline_cv['ds'], baseline_cv['y'], label='Actual', alpha=0.7)
    plt.plot(baseline_cv['ds'], baseline_cv['yhat'], label='Baseline', alpha=0.7)
    plt.title('Baseline CV Performance')
    plt.legend()
    plt.xticks(rotation=45)
else:
    plt.subplot(2, 2, 2)
    plt.text(0.5, 0.5, 'Baseline CV unavailable', ha='center', va='center')
    plt.axis('off')

if optimized_cv is not None:
    plt.subplot(2, 2, 3)
    plt.plot(optimized_cv['ds'], optimized_cv['y'], label='Actual', alpha=0.7)
    plt.plot(optimized_cv['ds'], optimized_cv['yhat'], label='Optimized', alpha=0.7)
    plt.title('Optimized CV Performance')
    plt.legend()
    plt.xticks(rotation=45)
else:
    plt.subplot(2, 2, 3)
    plt.text(0.5, 0.5, 'Optimized CV unavailable', ha='center', va='center')
    plt.axis('off')

plt.subplot(2, 2, 4)
plt.plot(test_data['ds'], test_data['y'], label='Actual Test', alpha=0.7)
plt.plot(test_predictions['ds'], test_predictions['yhat'], label='Optimized Forecast', alpha=0.7)
plt.title('Test Set Performance')
plt.legend()
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()